In [ ]:
import sys
import os

sys.path.append(os.path.abspath('../util'))

from stimulus_Loader import StimulusProcessor
from SIC import SICModelTorch
from BrainShow import BrainShow

In [ ]:
import os
import numpy as np
from tqdm import tqdm
from datetime import datetime
import csv
import torch

# ==================================================
# Stimulus NPZ
# ==================================================
NPZ_PATH = "../stimulus/VideoNPZ/video_stimulus.npz"

if not os.path.exists(NPZ_PATH):
    raise RuntimeError("❌ stimulus npz not found")

print("Loading stimulus...")

data = np.load(NPZ_PATH)
stimulus = data["frames"]     # shape (T,41,82)

print("Stimulus shape:", stimulus.shape)


# ==================================================
# Initialize LMC model
# ==================================================
print("\nInitializing LMCs model...")

SIC_model = SICModelTorch(
    matrix_dir="neuron_matrices",
    output_dir="../results/responses/video_stimulus",
    t_step=10,
    rate=100,
    device=torch.device("cuda:0")
)


# ==================================================
# Load neuron IDs
# ==================================================
def read_all_neuron_ids(filename='../data/classification.txt'):
    neuron_ids = []

    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)

            for row in reader:
                if 'root_id' in row and row['root_id'].strip():
                    try:
                        neuron_ids.append(int(row['root_id'].strip()))
                    except ValueError:
                        continue

    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")

    return neuron_ids


print("Loading neuron weights...")

neuron_ids = read_all_neuron_ids()
print(f"Found {len(neuron_ids)} neurons")

weights = SIC_model.load_weights(neuron_ids,normalize=True)

print("Weights loaded successfully")


# ==================================================
# Calculate response
# ==================================================
print("\nCalculating LMC responses...")

SIC_model.calculate_response_baseline(
    stimulus,
    weights,
    responce_threshold=0,
    stim_name="video_stimulus"
)


# ==================================================
# Summary
# ==================================================
print("\n" + "=" * 50)
print("RESPONSE CALCULATION COMPLETE")
print("=" * 50)
print(f"Stimulus file: {NPZ_PATH}")
print(f"Stimulus shape: {stimulus.shape}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

In [ ]:
import os
import glob
import numpy as np
import csv


def read_neuron_ids_by_groups(groups, filename='../data/names.txt'):
    """
    Read neuron root_ids whose group contains ANY of the given group keys.
    
    groups:
        - str: 'LO'
        - list/tuple/set: ['LO', 'ME']
    """
    if isinstance(groups, str):
        groups = [groups]

    group_keys = {g.lower() for g in groups}
    neuron_ids = []

    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'group' not in row or 'root_id' not in row:
                    continue

                group_parts = row['group'].lower().split('.')

                if any(key in group_parts for key in group_keys):
                    try:
                        neuron_ids.append(int(row['root_id']))
                    except ValueError:
                        continue

    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
    except Exception as e:
        print(f"Error reading file: {str(e)}")

    return neuron_ids

def read_all_neuron_ids(filename='../data/classification.txt'):
    neuron_ids = []

    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)

            for row in reader:
                if 'root_id' in row and row['root_id'].strip():
                    try:
                        neuron_ids.append(int(row['root_id'].strip()))
                    except ValueError:
                        continue

    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")

    return neuron_ids

neuron_ids = read_all_neuron_ids()  
brain_viz = BrainShow(
    swc_path='../data/sk_lod1_783_healed'
)
responses_path='./output/RealWorld/video_stimulus.npz'
npz_filename = os.path.basename(responses_path).replace(".npz", "")

output_dir = f'./output/RealWorld/{npz_filename}'
    
os.makedirs(output_dir, exist_ok=True)

brain_viz.plot_frames(
    neuron_ids=neuron_ids,
    output_dir=output_dir,
    responses_path=responses_path,
    dpi=600,
    frame_interval=4,
    video_fps=10,
    stimulus_path="../stimulus/autofly.mp4"
    )